# Zarr Basics

In [ ]:
import numpy as np
import zarr

zarr_root = "demo.zarr"

## Creating an array and writing data

In [ ]:
z = zarr.create_array(store=zarr_root, shape=(10000, 10000), chunks=(1000, 1000), dtype='int32')
print(z)

In [ ]:
# There is now a zarr.json file present
!cat demo.zarr/zarr.json

In [ ]:
print(f'(0,0)={z[0,0]}, (1500,2500)={z[1500,2500]}')

The only file written is `zarr.json`. We can still get values for any point in the array. They're all zero.

Setting values is easy

In [ ]:
z[1500,2500]= 999
print(f'(0,0)={z[0,0]}, (1500,2500)={z[1500,2500]}')

In [ ]:
!find demo.zarr -type f

The value is set as expected. And there is a new file that was created at path `c/1/2`. 
Why is it `1/2`?

Can also set with numpy slicing syntax:

In [ ]:
z[5,:] = np.arange(10000)

Before we check, guess what new files (if any) will be present.

In [ ]:
!find demo.zarr -type f | sort

## Read the data back

In [ ]:
z_read = zarr.open_array(zarr_root, mode='r')
z_read[5,:]

Hooray, the data that we wrote is what we read!

In [ ]:
z_sharded = zarr.create_array(
    store='demo_sharded.zarr', 
    shape=(10000, 10000), 
    shards=(5000, 5000), 
    chunks=(500, 500),
    dtype='int32')

print(z_sharded)

In [ ]:
z_sharded[1500,2500]= 999
z_sharded[5,:] = np.arange(10000)

In [ ]:
!find demo_sharded.zarr -type f | sort

## Custom metadata

Custom metadata go under the `attributes` key in `zarr.json`.  Right now there should be nothing there.:

In [ ]:
!jq '.attributes' demo.zarr/zarr.json

In [ ]:
z.attrs['says'] = {'dog':'woof', 'cat':'meow', 'cow':'moo'}

But after setting `z.attrs`, we get the metadata we set.

In [ ]:
!jq '.attributes' demo.zarr/zarr.json

## Exercise

Create a zarr array with a different `fill_value`. Make sure that's the value you get back for missing chunks.
Some [relevant documentation](https://zarr.readthedocs.io/en/stable/api/zarr/array/#zarr.Array.fill_value).